In [ ]:
!pip install openai langdetect googletrans==4.0.0-rc1 gTTS SpeechRecognition pygame


In [ ]:
import os
import time
import speech_recognition as sr
from googletrans import Translator
from gtts import gTTS
from langdetect import detect
from openai import OpenAI

# Optional: audio output
try:
    import pygame
    pygame.mixer.init()
    AUDIO_READY = True
except Exception:
    AUDIO_READY = False

recognizer = sr.Recognizer()
translator = Translator()


In [ ]:
# Safer key setup (do NOT hardcode your real key in git)
# Option A (recommended): set OPENAI_API_KEY in terminal before running notebook.
# Option B (temporary for this session): uncomment next 3 lines and paste key once.
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI API key: ")

if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is not set yet. Set it in terminal or with getpass() above.")

# Optional: if RAGDataset.txt exists, we use it as extra context
extra_context = ""
if os.path.exists("RAGDataset.txt"):
    with open("RAGDataset.txt", "r", encoding="utf-8") as f:
        extra_context = f.read().strip()


In [ ]:
def take_input():
    """Try microphone first, then text fallback."""
    try:
        with sr.Microphone() as source:
            print("Listening...")
            recognizer.adjust_for_ambient_noise(source)
            audio = recognizer.listen(source, timeout=8, phrase_time_limit=10)
        text = recognizer.recognize_google(audio)
        print("You said:", text)
        return text
    except Exception:
        return input("Type your question: ").strip()


def speak_response(text, lang):
    if not AUDIO_READY:
        print("(Audio not available in this environment)")
        return

    file_name = "response_voice.mp3"
    gTTS(text=text, lang=lang, slow=False).save(file_name)
    pygame.mixer.Sound(file_name).play()
    while pygame.mixer.get_busy():
        time.sleep(0.3)


def ask_assistant(user_text):
    input_lang = detect(user_text)
    english_text = translator.translate(user_text, dest="en").text

    client = OpenAI()
    system_msg = "Please answer clearly in simple language."
    if extra_context:
        system_msg += " Use this context if relevant: " + extra_context[:3000]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": english_text}
        ],
        temperature=0.5,
        max_tokens=180
    )

    answer_en = response.choices[0].message.content
    answer_local = translator.translate(answer_en, dest=input_lang).text

    print("Assistant:", answer_local)
    speak_response(answer_local, input_lang)


user_query = take_input()
if user_query:
    ask_assistant(user_query)
else:
    print("No input provided.")
